# Imports

In [87]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from utils import *
from valid_models import valid_models
from sklearn.exceptions import ConvergenceWarning
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from sklearn.model_selection import KFold, train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import BaggingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.svm import SVR, LinearSVR
from scipy.stats import uniform, randint

In [90]:
RANDOM_SEED = 1907
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [92]:
df=pd.read_csv("train.csv")
dfcopy = df.copy()

In [94]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75973 entries, 0 to 75972
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   carID           75973 non-null  int64  
 1   Brand           74452 non-null  object 
 2   model           74456 non-null  object 
 3   year            74482 non-null  float64
 4   price           75973 non-null  int64  
 5   transmission    74451 non-null  object 
 6   mileage         74510 non-null  float64
 7   fuelType        74462 non-null  object 
 8   tax             68069 non-null  float64
 9   mpg             68047 non-null  float64
 10  engineSize      74457 non-null  float64
 11  paintQuality%   74449 non-null  float64
 12  previousOwners  74423 non-null  float64
 13  hasDamage       74425 non-null  float64
dtypes: float64(8), int64(2), object(4)
memory usage: 8.1+ MB


In [96]:
num_cols = ['year', 'mileage', 'tax', 'mpg', 'engineSize']
cat_cols = ['Brand', 'model', 'transmission']
int_cols = ['year']
float_cols = ['mileage', 'tax', 'mpg', 'engineSize']

drop_cols = ['paintQuality%', 'previousOwners', 'fuelType']

In [98]:
X=clean_df(df, valid_models, cat_cols)
X, y = separar_y(X)

In [99]:
drop_cols = ['paintQuality%', 'previousOwners', 'fuelType']
X = X.drop(columns=drop_cols)

In [103]:
"""X_train, X_val, y_train, y_val = train_test_split(X,y, test_size = 0.3, random_state = RANDOM_SEED, shuffle = True)"""

'X_train, X_val, y_train, y_val = train_test_split(X,y, test_size = 0.3, random_state = RANDOM_SEED, shuffle = True)'

In [106]:
X_test = pd.read_csv("test.csv")
X_test = clean_df(X_test, valid_models, cat_cols)
X_test = X_test.drop(columns=drop_cols)

In [108]:
encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown="ignore")

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

scaler = StandardScaler()
kf = KFold(n_splits=7, shuffle=True, random_state=RANDOM_SEED)

In [110]:
def subsample_xy(X, y, n_samples=15000, random_state=1907):
    if len(X) <= n_samples:
        return X, y
    X_sub = X.sample(n=n_samples, random_state=random_state)
    return X_sub, y.loc[X_sub.index]

In [113]:
"""svr_model_class = SVR
svr_params = {'kernel': 'rbf', 'C': 1.0, 'epsilon': 0.1, 'gamma': 'scale', 'max_iter': 10000}

print("--- Benchmarking Traditional SVR (RBF Kernel) ---")
results_svr = avg_score(
    method=kf,
    X=X,
    y=y,
    int_cols=int_cols,
    float_cols=float_cols,
    model_class=svr_model_class,
    model_params=svr_params,
    scaler=scaler,
    encoder=encoder,
    cat_cols=cat_cols,
    encoding_type="ohe"
)
print(f"SVR (RBF) Results: Val MAE = {np.mean(results_svr['mae_val']):.4f}, Val R² = {np.mean(results_svr['r2_val']):.4f}\n")

linear_svr_model_class = LinearSVR
linear_svr_params = {'epsilon': 0.0, 'C': 1.0, 'loss': 'epsilon_insensitive', 'max_iter': 50000} # Increased max_iter for convergence

print("--- Benchmarking Linear SVR ---")
results_linear_svr = avg_score(
    method=kf,
    X=X,
    y=y,
    int_cols=int_cols,
    float_cols=float_cols,
    model_class=linear_svr_model_class,
    model_params=linear_svr_params,
    scaler=scaler,
    encoder=encoder,
    cat_cols=cat_cols,
    encoding_type="ohe"
)
print(f"Linear SVR Results: Val MAE = {np.mean(results_linear_svr['mae_val']):.4f}, Val R² = {np.mean(results_linear_svr['r2_val']):.4f}\n")



bagged_svr_model_class = BaggingRegressor
bagged_svr_params = {
    'estimator': SVR(kernel='rbf', C=1.0, epsilon=0.1, gamma='scale', max_iter=1000),
    'n_estimators': 10,
    'random_state': RANDOM_SEED,
    'n_jobs': -1
}

print("--- Benchmarking Bagged SVR (RBF Kernel) ---")

results_bagged_svr = avg_score(
    method=kf,
    X=X,
    y=y,
    int_cols=int_cols,
    float_cols=float_cols,
    model_class=bagged_svr_model_class,
    model_params=bagged_svr_params,
    scaler=scaler,
    encoder=encoder,
    cat_cols=cat_cols,
    encoding_type="ohe"
)
print(f"Bagged SVR Results: Val MAE = {np.mean(results_bagged_svr['mae_val']):.4f}, Val R² = {np.mean(results_bagged_svr['r2_val']):.4f}\n")"""

'svr_model_class = SVR\nsvr_params = {\'kernel\': \'rbf\', \'C\': 1.0, \'epsilon\': 0.1, \'gamma\': \'scale\', \'max_iter\': 10000}\n\nprint("--- Benchmarking Traditional SVR (RBF Kernel) ---")\nresults_svr = avg_score(\n    method=kf,\n    X=X,\n    y=y,\n    int_cols=int_cols,\n    float_cols=float_cols,\n    model_class=svr_model_class,\n    model_params=svr_params,\n    scaler=scaler,\n    encoder=encoder,\n    cat_cols=cat_cols,\n    encoding_type="ohe"\n)\nprint(f"SVR (RBF) Results: Val MAE = {np.mean(results_svr[\'mae_val\']):.4f}, Val R² = {np.mean(results_svr[\'r2_val\']):.4f}\n")\n\nlinear_svr_model_class = LinearSVR\nlinear_svr_params = {\'epsilon\': 0.0, \'C\': 1.0, \'loss\': \'epsilon_insensitive\', \'max_iter\': 50000} # Increased max_iter for convergence\n\nprint("--- Benchmarking Linear SVR ---")\nresults_linear_svr = avg_score(\n    method=kf,\n    X=X,\n    y=y,\n    int_cols=int_cols,\n    float_cols=float_cols,\n    model_class=linear_svr_model_class,\n    model_par

In [115]:
"""experiments = [
    # 1. RBF SVR – subsampled – no log
    {
        "name": "rbf_svr_sub_no_log",
        "model_class": SVR,
        "model_params": {'kernel': 'rbf', 'C': 1.0, 'epsilon': 0.1, 'gamma': 'scale', 'max_iter': 10000},
        "use_log": False,
        "subsample": True
    },

    # 2. RBF SVR – subsampled – log(price)
    {
        "name": "rbf_svr_sub_log",
        "model_class": SVR,
        "model_params": {'kernel': 'rbf', 'C': 1.0, 'epsilon': 0.1, 'gamma': 'scale', 'max_iter': 10000},
        "use_log": True,
        "subsample": True
    },

    # 3. Linear SVR – no log
    {
        "name": "linear_svr_no_log",
        "model_class": LinearSVR,
        "model_params": {'epsilon': 0.0, 'C': 52.983, 'loss': 'epsilon_insensitive', 'max_iter': 50000},
        "use_log": False,
        "subsample": False
    },

    # 4. Linear SVR – log(price)
    {
        "name": "linear_svr_log",
        "model_class": LinearSVR,
        "model_params": {'epsilon': 0.0, 'C': 52.983, 'loss': 'epsilon_insensitive', 'max_iter': 50000},
        "use_log": True,
        "subsample": False
    },

    # 5. Bagged Linear SVR – no log
    {
        "name": "bagged_linear_svr_no_log",
        "model_class": BaggingRegressor,
        "model_params": {
            'estimator': LinearSVR(C=52.983, epsilon=0.0, max_iter=50000, loss='epsilon_insensitive'),
            'n_estimators': 10,
            'random_state': RANDOM_SEED,
            'n_jobs': -1
        },
        "use_log": False,
        "subsample": False
    },
    
    # 6. Bagged Linear SVR – log(price)
    {
        "name": "bagged_linear_svr_log",
        "model_class": BaggingRegressor,
        "model_params": {
            'estimator': LinearSVR(C=52.983, epsilon=0.0, max_iter=50000, loss='epsilon_insensitive'),
            'n_estimators': 10,
            'random_state': RANDOM_SEED,
            'n_jobs': -1
        },
        "use_log": True,
        "subsample": False
    }
]"""

'experiments = [\n    # 1. RBF SVR – subsampled – no log\n    {\n        "name": "rbf_svr_sub_no_log",\n        "model_class": SVR,\n        "model_params": {\'kernel\': \'rbf\', \'C\': 1.0, \'epsilon\': 0.1, \'gamma\': \'scale\', \'max_iter\': 10000},\n        "use_log": False,\n        "subsample": True\n    },\n\n    # 2. RBF SVR – subsampled – log(price)\n    {\n        "name": "rbf_svr_sub_log",\n        "model_class": SVR,\n        "model_params": {\'kernel\': \'rbf\', \'C\': 1.0, \'epsilon\': 0.1, \'gamma\': \'scale\', \'max_iter\': 10000},\n        "use_log": True,\n        "subsample": True\n    },\n\n    # 3. Linear SVR – no log\n    {\n        "name": "linear_svr_no_log",\n        "model_class": LinearSVR,\n        "model_params": {\'epsilon\': 0.0, \'C\': 52.983, \'loss\': \'epsilon_insensitive\', \'max_iter\': 50000},\n        "use_log": False,\n        "subsample": False\n    },\n\n    # 4. Linear SVR – log(price)\n    {\n        "name": "linear_svr_log",\n        "model_

**A traditional SVR is not viable since MAE too high and computationally heavy**

In [119]:

C_grid = np.logspace(-1, 3, 10)  # 0.1 → 1000
epsilon_grid = [0.0, 0.01, 0.05]  # optional, small set


pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearSVR(max_iter=50000, loss='epsilon_insensitive', random_state=RANDOM_SEED))
])

param_grid = {
    'model__C': C_grid,
    'model__epsilon': epsilon_grid
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=kf,
    n_jobs=-1,
    verbose=2
)

y_used = np.log1p(y)  # keep log(price)
grid_search.fit(X, y_used)

print("Best parameters found:", grid_search.best_params_)
print("Best CV MAE:", -grid_search.best_score_)  # negative because sklearn uses neg MAE

Fitting 7 folds for each of 30 candidates, totalling 210 fits


/opt/anaconda3/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[CV] END ...................model__C=0.1, model__epsilon=0.0; total time=   5.4s
[CV] END ...................model__C=0.1, model__epsilon=0.0; total time=   5.6s


/opt/anaconda3/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[CV] END ...................model__C=0.1, model__epsilon=0.0; total time=   5.8s


KeyboardInterrupt: 

In [74]:
experiments = [

        # 4. Linear SVR – log(price)
    {
        "name": "linear_svr_log",
        "model_class": LinearSVR,
        "model_params": {'epsilon': 0.01, 'C': 10, 'loss': 'epsilon_insensitive', 'max_iter': 50000},
        "use_log": True,
        "subsample": False
    }
]

""" 
# 1. RBF SVR – subsampled – no log
    {
        "name": "rbf_svr_sub_no_log",
        "model_class": SVR,
        "model_params": {'kernel': 'rbf', 'C': 1.0, 'epsilon': 0.1, 'gamma': 'scale', 'max_iter': 5000},
        "use_log": False,
        "subsample": True
    },
    # 2. RBF SVR – subsampled – log(price)
    {
        "name": "rbf_svr_sub_log",
        "model_class": SVR,
        "model_params": {'kernel': 'rbf', 'C': 1.0, 'epsilon': 0.1, 'gamma': 'scale', 'max_iter': 5000},
        "use_log": True,
        "subsample": True
    },
    # 3. Linear SVR – no log
    {
        "name": "linear_svr_no_log",
        "model_class": LinearSVR,
        "model_params": {'epsilon': 0.0, 'C': 52.983, 'loss': 'epsilon_insensitive', 'max_iter': 50000},
        "use_log": False,
        "subsample": False
    },"""


"""# 5. Bagged Linear SVR – no log
    {
        "name": "bagged_linear_svr_no_log",
        "model_class": BaggingRegressor,
        "model_params": {
            'estimator': LinearSVR(epsilon=0.0, C=52.983, max_iter=50000),
            'n_estimators': 10,
            'random_state': RANDOM_SEED,
            'n_jobs': -1
        },
        "use_log": False,
        "subsample": False
    },
    # 6. Bagged Linear SVR – log(price)
    {
        "name": "bagged_linear_svr_log",
        "model_class": BaggingRegressor,
        "model_params": {
            'estimator': LinearSVR(epsilon=0.0, C=52.983, max_iter=50000),
            'n_estimators': 10,
            'random_state': RANDOM_SEED,
            'n_jobs': -1
        },
        "use_log": True,
        "subsample": False
    }"""

'# 5. Bagged Linear SVR – no log\n    {\n        "name": "bagged_linear_svr_no_log",\n        "model_class": BaggingRegressor,\n        "model_params": {\n            \'estimator\': LinearSVR(epsilon=0.0, C=52.983, max_iter=50000),\n            \'n_estimators\': 10,\n            \'random_state\': RANDOM_SEED,\n            \'n_jobs\': -1\n        },\n        "use_log": False,\n        "subsample": False\n    },\n    # 6. Bagged Linear SVR – log(price)\n    {\n        "name": "bagged_linear_svr_log",\n        "model_class": BaggingRegressor,\n        "model_params": {\n            \'estimator\': LinearSVR(epsilon=0.0, C=52.983, max_iter=50000),\n            \'n_estimators\': 10,\n            \'random_state\': RANDOM_SEED,\n            \'n_jobs\': -1\n        },\n        "use_log": True,\n        "subsample": False\n    }'

In [76]:
all_results = {}

for exp in experiments:
    print(f"\n--- Running {exp['name']} ---")

    X_run, y_run = X, y
    if exp["subsample"]:
        X_run, y_run = subsample_xy(X, y, n_samples=15000, random_state=RANDOM_SEED)

    y_used = np.log1p(y_run) if exp["use_log"] else y_run

    results = avg_score(
        method=kf,
        X=X_run,
        y=y_used,
        int_cols=int_cols,
        float_cols=float_cols,
        cat_cols=cat_cols,
        model_class=exp["model_class"],
        model_params=exp["model_params"],
        scaler=scaler,
        encoder=encoder,
        encoding_type="ohe"
    )

    all_results[exp["name"]] = results

    print(
        f"Val MAE = {np.mean(results['mae_val']):.4f}, "
        f"Val R² = {np.mean(results['r2_val']):.4f}"
    )


--- Running linear_svr_log ---
Average Train MAE: 0.1202
Average Val MAE:   0.1208
Average Train R²:  0.8967
Average Val R²:    0.8959
Val MAE = 0.1208, Val R² = 0.8959


In [ ]:
"""--- Running rbf_svr_sub_no_log ---
Average Train MAE: 6542.2081
Average Val MAE:   6542.7466
Average Train R²:  -0.0268
Average Val R²:    -0.0266
Val MAE = 6542.7466, Val R² = -0.0266

--- Running rbf_svr_sub_log ---
Average Train MAE: 0.0840
Average Val MAE:   0.0966
Average Train R²:  0.9558
Average Val R²:    0.9333
Val MAE = 0.0966, Val R² = 0.9333

--- Running linear_svr_no_log ---
Average Train MAE: 2616.5166
Average Val MAE:   2622.0463
Average Train R²:  0.7534
Average Val R²:    0.7531
Val MAE = 2622.0463, Val R² = 0.7531

--- Running linear_svr_log ---
Average Train MAE: 0.1202
Average Val MAE:   0.1208
Average Train R²:  0.8965
Average Val R²:    0.8957
Val MAE = 0.1208, Val R² = 0.8957

Best parameters found: {'model__C': 5.994842503189409, 'model__epsilon': 0.01}
Best CV MAE: 0.11125466628711664

--- Running linear_svr_log ---
Average Train MAE: 0.1202
Average Val MAE:   0.1208
Average Train R²:  0.8967
Average Val R²:    0.8959
Val MAE = 0.1208, Val R² = 0.8959

"""

In [70]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", scaler)
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", encoder)
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, int_cols + float_cols),
    ("cat", categorical_transformer, cat_cols)
])

In [72]:
os.makedirs("kaggle", exist_ok=True)

for exp in experiments:
    print(f"\n--- Training final model for {exp['name']} ---")

    y_train_final = np.log1p(y) if exp["use_log"] else y
    
    model_cls = exp["model_class"]
    model_params = exp["model_params"]
    if model_cls == BaggingRegressor:
            model = model_cls(**model_params)
    else:
            model = model_cls(**model_params)
    
    pipeline = Pipeline([
                ('preprocessor', preprocessor),
                ('model', exp["model_class"](**exp["model_params"]))
            ])

    pipeline.fit(X, y_train_final)

    y_pred = pipeline.predict(X_test)

    if exp["use_log"]:
        y_pred = np.expm1(y_pred)

    submission = pd.DataFrame({
        "carID": X_test.index,
        "price": y_pred
    })

    submission_path = f"kaggle/{exp['name']}.csv"
    submission.to_csv(submission_path, index=False)

    print(f"Saved → {submission_path}")


--- Training final model for linear_svr_log ---
Saved → kaggle/linear_svr_log.csv


In [79]:
"""
--- Traditional SVR (RBF Kernel) ---
Average Train MAE: 6443.1240
Average Val MAE:   6443.2249
Average Train R²:  0.1043
Average Val R²:    0.1043
SVR (RBF) Results: Val MAE = 6443.2249, Val R² = 0.1043

--- Linear SVR ---
Average Train MAE: 3371.3627
Average Val MAE:   3372.5903
Average Train R²:  0.6407
Average Val R²:    0.6407
Linear SVR Results: Val MAE = 3372.5903, Val R² = 0.6407

--- Bagged SVR (RBF Kernel) ---
Average Train MAE: 13559.7977
Average Val MAE:   13560.1879
Average Train R²:  -1.4308
Average Val R²:    -1.4306
Bagged SVR Results: Val MAE = 13560.1879, Val R² = -1.4306

"""

'\n--- Traditional SVR (RBF Kernel) ---\nAverage Train MAE: 6443.1240\nAverage Val MAE:   6443.2249\nAverage Train R²:  0.1043\nAverage Val R²:    0.1043\nSVR (RBF) Results: Val MAE = 6443.2249, Val R² = 0.1043\n\n--- Linear SVR ---\nAverage Train MAE: 3371.3627\nAverage Val MAE:   3372.5903\nAverage Train R²:  0.6407\nAverage Val R²:    0.6407\nLinear SVR Results: Val MAE = 3372.5903, Val R² = 0.6407\n\n--- Bagged SVR (RBF Kernel) ---\nAverage Train MAE: 13559.7977\nAverage Val MAE:   13560.1879\nAverage Train R²:  -1.4308\nAverage Val R²:    -1.4306\nBagged SVR Results: Val MAE = 13560.1879, Val R² = -1.4306\n\n'

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearSVR(
        loss='epsilon_insensitive',
        max_iter=50000,
        random_state=RANDOM_SEED
    ))
])

param_dist = {
    'model__C': loguniform(0.5, 50),   # around your optimum
    'model__epsilon': [0.005, 0.01, 0.02]
}

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=25,                         # enough for LinearSVR
    scoring='neg_mean_absolute_error',
    cv=kf,
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbose=2
)

y_used = np.log1p(y)
random_search.fit(X, y_used)

print("Best params:", random_search.best_params_)
print("Best CV MAE:", -random_search.best_score_)